# Combining Models: Voting, Averaging and Stacking

**Notebook 1 of 3.** Every model sees the data through its own assumptions, so every model
has its own blind spots. Combining several lets one model's mistakes be outvoted by the
rest — under conditions that are easy to state and easy to violate.

## 1. The data

`data/census.csv`: 45,222 records from the 1994 US census, five numeric columns and eight
text ones, with `income` saying whether the person earned more than $50,000. One-hot
encoding the text columns turns 13 columns into 103.

Just under a quarter earned more than $50,000, and that imbalance picks the metric.
Answering `<=50K` to everyone is **75.2% accurate** and useless, so this notebook scores on
**F1**, the harmonic mean of precision and recall, which that answer scores **0** on.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier, VotingClassifier, StackingClassifier

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# one seed for every split and every model, so the numbers below are reproducible
RSEED = 42

In [ ]:
census = pd.read_csv("data/census.csv")

categorical = [
    "workclass", "education_level", "marital-status", "occupation",
    "relationship", "race", "sex", "native-country",
]

y = (census["income"] == ">50K").astype(int)

# astype(float) matters: get_dummies returns booleans, and a mixed frame becomes an
# object array, which makes every fit below several times slower
X = pd.get_dummies(census.drop(columns=["income"]), columns=categorical).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED, stratify=y
)

print(f"{X.shape[1]} encoded columns, {len(X_train)} training rows, {len(X_test)} test rows")
print(f"earning >50K: {y_test.mean():.4f} of the test rows")
print(f"answering <=50K to everyone: accuracy {(y_test == 0).mean():.4f}, F1 0.0000")

## 2. Five models, and two of them are new

The members are deliberately mixed, because models that reason alike fail alike.

- **Logistic regression** runs a weighted sum of the columns through an S-curve, so its
  boundary is a flat plane.
- **K-nearest neighbors** does no fitting: it stores the training rows and labels a new
  person by a majority vote of the `k` closest, `k=5` by default. Distance is measured in
  the units of the columns, so a column in thousands would drown out one in years — the
  columns must be scaled first.
- **Gaussian naive Bayes** applies Bayes' rule to get each class's probability given the
  row. Doing that exactly across 103 columns is impossible, so it assumes every column is
  independent of the others *given the class* and multiplies their contributions. That
  assumption is false here — education and occupation are related — and it is what
  **naive** names. One normal curve per column per class, no iteration.
- **A decision tree** asks yes/no questions until its leaves are pure.
- **AdaBoost on ten stumps** adds ten depth-1 trees in sequence, each told where the ones
  before it went wrong.

The first two go in a `StandardScaler` pipeline. The other three do not need one: a tree
compares a column against a threshold and naive Bayes fits each column separately.

In [ ]:
def members():
    """Build a fresh, unfitted list of the five ensemble members.

    A function rather than a list, because every section below needs its own unfitted
    copies: handing the same fitted object to two ensembles would let one refit the other.

    Returns:
        list: (name, estimator) pairs, in the order used throughout the notebook.
    """
    return [
        ("lr", make_pipeline(StandardScaler(), LogisticRegression(random_state=RSEED))),
        ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_jobs=-1))),
        ("dt", DecisionTreeClassifier(random_state=RSEED)),
        ("nb", GaussianNB()),
        # estimator= is the weak learner; scikit-learn 1.9 renamed it from base_estimator
        ("ada", AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                                   n_estimators=10, random_state=RSEED)),
    ]


NAMES = [name for name, _ in members()]

In [ ]:
# fit each member once and keep both its labels and its probabilities: the later
# sections combine these same five numbers in four different ways
labels, probabilities = {}, {}

for name, model in members():
    model.fit(X_train, y_train)
    labels[name] = model.predict(X_test)
    # column 1 of predict_proba is P(>50K), which is what soft voting averages
    probabilities[name] = model.predict_proba(X_test)[:, 1]

scores = pd.DataFrame(
    {
        "F1": [f1_score(y_test, labels[n]) for n in NAMES],
        "precision": [precision_score(y_test, labels[n]) for n in NAMES],
        "recall": [recall_score(y_test, labels[n]) for n in NAMES],
        "accuracy": [accuracy_score(y_test, labels[n]) for n in NAMES],
    },
    index=NAMES,
)
print(scores.to_string(float_format="%.4f"))

Naive Bayes has the **best F1 and the worst accuracy**. Its independence assumption makes
it overconfident, so it says `>50K` far more often: recall 0.79 against AdaBoost's 0.49,
bought with precision 0.57 against AdaBoost's 0.78.

Near-opposites, and that is the useful part — they are wrong about **different people**.

## 3. The two conditions

Combining helps when both hold, and not otherwise:

1. **every member beats guessing** — the worst scores F1 0.598 against the do-nothing
   answer's 0;
2. **the members are wrong about different rows** — if all five err together, a vote
   changes nothing.

The second is something you can count.

In [ ]:
# how often does each pair of members hand in a different answer?
disagreement = pd.DataFrame(
    [[(labels[a] != labels[b]).sum() for b in NAMES] for a in NAMES],
    index=NAMES, columns=NAMES,
)
print(f"out of {len(y_test)} test rows:\n")
print(disagreement.to_string())

In [ ]:
# wrong[i, j] is True when member i gets test row j wrong
wrong = np.stack([labels[n] != y_test.values for n in NAMES])
votes_wrong = wrong.sum(axis=0)          # how many of the five missed each row

print(f"rows no member gets wrong:       {(votes_wrong == 0).sum()}")
print(f"rows exactly one gets wrong:     {(votes_wrong == 1).sum()}")
# a majority vote survives a minority being wrong, and loses when three or more are
print(f"rows a minority (1-2) get wrong: {((votes_wrong >= 1) & (votes_wrong <= 2)).sum()}")
print(f"rows a majority (3+) get wrong:  {(votes_wrong >= 3).sum()}")
print(f"rows every member gets wrong:    {(votes_wrong == 5).sum()}")

The members disagree on 662 to 1,780 of the 9,045 test rows, and the extremes land where
you would expect: naive Bayes against the tree at 1,780, logistic regression against
AdaBoost at only 662 — the two that already think alike.

The second block is the arithmetic a vote runs on:

- **1,830 rows** where one or two members are wrong, which a vote **fixes**;
- **1,374 rows** where three or more are wrong, which it **cannot**;
- **450 rows** every member gets wrong.

A vote trades the first against the second. That is the whole mechanism.

## 4. Max voting

In **max voting**, also called hard voting, each member predicts a label and the ensemble
returns whichever label most of them chose. A show of hands.

In [ ]:
# voting="hard" counts labels; n_jobs=-1 fits the five members in parallel
hard = VotingClassifier(estimators=members(), voting="hard", n_jobs=-1)
hard.fit(X_train, y_train)
hard_pred = hard.predict(X_test)

print(f"hard voting   F1 {f1_score(y_test, hard_pred):.4f}   "
      f"precision {precision_score(y_test, hard_pred):.4f}   "
      f"recall {recall_score(y_test, hard_pred):.4f}")
print(f"best member   F1 {scores['F1'].max():.4f}  ({scores['F1'].idxmax()})")

**0.6713 against the best member's 0.6652** — a real gain, and a small one.

Hard voting throws something away: a member that was 51% sure counts as much as one that
was 99% sure. The probabilities are already computed, so the next rule uses them.

## 5. Soft voting

**Soft voting** averages the predicted probabilities and takes the highest, so a confident
member pulls the result further than an unsure one.

> **Exercise 1.** Build a `VotingClassifier` over the same five members that averages
> probabilities instead of counting labels, and print its F1, precision and recall as the
> cell above does. Which of precision and recall moved?

<details><summary>
Click here for a hint...
</summary>

`voting="soft"` is the only argument that changes.

</details>

In [ ]:
# your code: a soft-voting ensemble over members(), fitted on the training rows, with
# its F1, precision and recall on the test rows printed

<details><summary>
Click here to compare your answer...
</summary>

| | F1 | precision | recall |
| :-- | --: | --: | --: |
| best member (nb) | 0.6652 | 0.5735 | 0.7917 |
| hard voting | 0.6713 | 0.7239 | 0.6258 |
| **soft voting** | **0.6809** | 0.6881 | 0.6740 |

**Recall moved**, 0.6258 to 0.6740, for 3.6 points of precision. Hard voting needs three of
five members to commit before it will; soft voting lets one confident member outweigh two
unsure ones, so borderline high earners get caught that a show of hands missed.

</details>

## 6. Averaging by hand

Soft voting is an average of probabilities, so it can be written out directly.

In [ ]:
# the same five probability columns the members produced in section 2
P = np.stack([probabilities[n] for n in NAMES])

# mean across members, then the usual 0.5 cutoff. No rounding before the comparison:
# rounding each probability first would throw away everything between 0.5 and the edges
plain = (P.mean(axis=0) >= 0.5).astype(int)
print(f"plain average   F1 {f1_score(y_test, plain):.4f}")

**0.6809, identical to soft voting**, because it is the same calculation. Writing it out
shows what a fixed rule can and cannot do: the mean trusts all five members equally, and
nothing in it can notice that `nb` is better than `dt`.

Giving better members a larger say is possible — weight by each member's cross-validated
score rather than averaging flat — but on these five it changes almost nothing, because the
scores run only from 0.596 to 0.672 and weights built from five similar numbers come out
near-equal. The way to make the combination actually learn is to **fit** it, which is
section 7.

## 7. Stacking

Voting and averaging combine with a **fixed rule**, decided before any data was seen.
**Stacking** trains a second model, the **meta-model**, to learn the rule from the five
members' predictions.

The danger is leakage: a meta-model trained on predictions the members made about rows they
were fitted on would see five unrealistically good columns. `StackingClassifier` builds each
column from **out-of-fold** predictions instead.

In [ ]:
stack = StackingClassifier(
    estimators=members(),
    # a deliberately simple meta-model: with five input columns there is little to learn,
    # and a complex one would overfit them
    final_estimator=LogisticRegression(),
    n_jobs=-1,
)
stack.fit(X_train, y_train)
stack_pred = stack.predict(X_test)

print(f"stacking   F1 {f1_score(y_test, stack_pred):.4f}   "
      f"precision {precision_score(y_test, stack_pred):.4f}   "
      f"recall {recall_score(y_test, stack_pred):.4f}   "
      f"accuracy {accuracy_score(y_test, stack_pred):.4f}")
# section 5's numbers, quoted rather than recomputed: this cell has to run whether or not
# the exercise above has been done
print("soft vote  F1 0.6809   precision 0.6881   recall 0.6740   accuracy 0.8434")

# transform() returns the five out-of-fold columns the meta-model was fitted on. A
# coefficient only means something beside the spread of the column it multiplies
meta_columns = stack.transform(X_train)
spread = meta_columns.std(axis=0)
print("\n", pd.DataFrame(
    {
        "coefficient": stack.final_estimator_.coef_[0],
        "column spread": spread,
        "coefficient x spread": stack.final_estimator_.coef_[0] * spread,
    },
    index=NAMES,
).to_string(float_format="%.3f"), sep="")

**Stacking scores 0.6626 against soft voting's 0.6809, while reaching 0.8473 on accuracy
against 0.8434.** Both are true, and they are one fact: read precision and recall.

Stacking reaches precision 0.7326 and recall 0.6048, where naive Bayes alone reaches 0.5735
and 0.7917. The meta-model moved hard toward precision. That buys accuracy, since most
people earn `<=50K` and declining to commit is usually right, and it costs F1, which only
rewards finding high earners. **The meta-model minimizes log loss, not the reported metric.**

The table carries a trap worth naming. AdaBoost's raw coefficient is the largest at 4.43,
but its column spans only 0.159 to 0.767 where the others span 0 to 1 — a narrow column
needs a large coefficient to say the same thing. Multiplied by its spread, logistic
regression leads instead. **Coefficients compare only across columns of similar scale.**
Naive Bayes sits near the bottom either way.

## 8. One split is one number

Every score above came from a single train/test split, and the gaps are small — 0.0157
between soft voting and the best member. Five folds of the training rows say whether they
survive resampling.

In [ ]:
# five folds of the training rows, stratified so each fold keeps the 1-in-4 imbalance
cv = StratifiedKFold(5, shuffle=True, random_state=RSEED)

# the same members scored on folds rather than the single split. Allow about a minute:
# the stacking line cross-validates its own five members inside each of the five folds
for label, model in [
    ("nb alone", GaussianNB()),
    ("soft voting", VotingClassifier(estimators=members(), voting="soft", n_jobs=-1)),
    ("stacking", StackingClassifier(estimators=members(),
                                    final_estimator=LogisticRegression(), n_jobs=-1)),
]:
    fold_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
    print(f"{label:<12} mean {fold_f1.mean():.4f}   "
          f"folds {' '.join(f'{v:.4f}' for v in fold_f1)}")

**Soft voting 0.6809 against naive Bayes' 0.6724, ahead in all five folds.** The single
split was not flattering it.

Stacking is what changes. On the single split it scored 0.6626, below naive Bayes; across
folds it reaches 0.6731 against 0.6724, ahead in three of five. It does not lose to its best
member — it **ties** it. The single split overstated that gap just as it understated soft
voting's, which is why a difference this small is not worth believing from one split.

## Summary

- An ensemble needs members that **beat guessing** and are **wrong about different rows**:
  1,830 test rows had a minority wrong, which a vote fixes, against 1,374 with a majority
  wrong, which it cannot.
- **Hard voting** reached 0.6713; **soft voting** reached 0.6809, ahead of the best member
  in all five folds.
- **Stacking** learns the combination instead of fixing it. It traded recall for precision,
  0.6048 against naive Bayes' 0.7917, which buys accuracy and costs F1: its meta-model
  minimizes log loss, not the reported metric. Across folds it **ties** its best member.

## References & Further Reading

- [**scikit-learn: Ensembles**](https://scikit-learn.org/stable/modules/ensemble.html): Voting, stacking, bagging and boosting in one place.
- [**scikit-learn: VotingClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html): Hard and soft voting, including the `weights` argument section 6 applies by hand.
- [**scikit-learn: Naive Bayes**](https://scikit-learn.org/stable/modules/naive_bayes.html): Why it classifies well while estimating probabilities badly.
- [**scikit-learn: Nearest Neighbors**](https://scikit-learn.org/stable/modules/neighbors.html): Choosing `k`, the distance metrics, and why scaling is required.
- [**Wolpert, Stacked Generalization (1992)**](https://doi.org/10.1016/S0893-6080(05)80023-1): Training a model on other models' out-of-sample predictions.
- [**UCI: Adult**](https://archive.ics.uci.edu/dataset/2/adult): The census data, extracted by Barry Becker and Ronny Kohavi, licensed CC BY 4.0.